# 2. Lowest Common Ancestor of a Binary Tree
**Difficulty:** 🟡 Medium · **Topic:** Trees / DFS · **LeetCode:** https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree/

> **DevRev context:** given two items in a hierarchy — two sub-tasks, two comments in a thread, two nodes in a workflow — find their **deepest common parent**. Unlike a BST you *cannot* navigate by comparing values, because a general task tree has no ordering; you must actually search both subtrees.

## 💡 Concepts

**Core concept(s):** A single **DFS** that returns the LCA by asking each subtree "did you contain p or q?".

**Why it applies here:** In a general binary tree there's no ordering to steer by (that trick only works for a BST). So you search: if `p` and `q` are found in **different** subtrees of a node, that node is their lowest common ancestor. If both are in the same subtree, the answer is deeper in that side.

**Key intuition:** A node is the LCA when one target is found on its left and the other on its right (or the node itself is one of them).

---

### 📚 What is DFS (Depth-First Search)?
**DFS** explores one branch as deep as possible, then backtracks (usually via recursion). On a tree it visits every node once → **O(n)** time, **O(height)** stack.

### 📚 Why this differs from the BST version
In a **BST** you compare values to walk straight to the split point (`O(height)`). A **general** tree has no such order, so you must explore both children and combine their answers — `O(n)`.

---

**Prerequisite knowledge:**
- Recursion that returns a node up the call stack.
- The "found in different subtrees" combine rule.

## 📝 Problem

Given a binary tree and two node values `p` and `q` (both present, all values distinct), return the value
of their **lowest common ancestor** — the deepest node that has both `p` and `q` somewhere below it (a node
can be its own ancestor).

**Example**
```
        3
       / \
      5   1
     / \ / \
    6  2 0  8
      / \
     7   4
LCA(5, 1) = 3 ;  LCA(5, 4) = 5  (5 is an ancestor of 4)
```

> Two approaches, both `O(n)`: path-based (find each root-to-node path, compare) and a single recursive DFS.

In [ ]:
from typing import Optional, List
from collections import deque

class TreeNode:
    """A binary-tree node: a value plus up to two children."""
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def build_tree(values):
    """Build a tree from a level-order list (None = missing child)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0]); q = deque([root]); i = 1
    while q and i < len(values):
        node = q.popleft()
        if i < len(values):
            if values[i] is not None:
                node.left = TreeNode(values[i]); q.append(node.left)
            i += 1
        if i < len(values):
            if values[i] is not None:
                node.right = TreeNode(values[i]); q.append(node.right)
            i += 1
    return root

def build_balanced(n):
    """A balanced tree holding 1..n (height ~log n) — keeps recursion shallow for the benchmark."""
    def helper(lo, hi):
        if lo > hi:
            return None
        mid = (lo + hi) // 2
        node = TreeNode(mid)
        node.left = helper(lo, mid - 1)
        node.right = helper(mid + 1, hi)
        return node
    return helper(1, n)

### Approach 1 — Root-to-Node Paths (worst on space/clarity)

**Idea:** Find the path from the root to `p` and to `q`. The last node they share is the LCA.

**Time:** `O(n)`. **Space:** `O(n)` for the two paths.

In [ ]:
def lca_paths(root, p, q):
    def find_path(node, target, path):
        if not node:
            return False
        path.append(node)                      # tentatively take this node
        if node.val == target:
            return True                        # found it -> path is complete
        if find_path(node.left, target, path) or find_path(node.right, target, path):
            return True                        # found deeper on one side
        path.pop()                             # dead end -> backtrack this node off the path
        return False
    pp, qp = [], []
    find_path(root, p, pp)                      # root-to-p path
    find_path(root, q, qp)                      # root-to-q path
    lca = None
    for a, b in zip(pp, qp):                    # walk both paths together...
        if a is b:
            lca = a                             # ...they agree here -> still a common ancestor
        else:
            break                               # they diverge -> the last agreement was the LCA
    return lca.val if lca else None

### Approach 2 — Single Recursive DFS (optimal)

**Idea:** DFS returns a node if `p` or `q` (or their LCA) is found in that subtree. If a node gets a
non-null answer from **both** children, it is the split point → the LCA.

**Time:** `O(n)`. **Space:** `O(height)`.

In [ ]:
def lca_recursive(root, p, q):
    def dfs(node):
        if not node:
            return None
        if node.val == p or node.val == q:
            return node                        # this subtree contains a target -> report it up
        left = dfs(node.left)                  # search both sides
        right = dfs(node.right)
        if left and right:
            return node                        # p on one side, q on the other -> THIS is the LCA
        return left or right                   # otherwise bubble up whichever side found something
    ans = dfs(root)
    return ans.val if ans else None

In [ ]:
# Correctness check
root = build_tree([3, 5, 1, 6, 2, 0, 8, None, None, 7, 4])
tests = [(5, 1, 3), (5, 4, 5), (6, 4, 5), (7, 8, 3), (0, 8, 1)]
for p, q, exp in tests:
    a, b = lca_paths(root, p, q), lca_recursive(root, p, q)
    print(f"LCA({p}, {q}) -> paths={a}, recursive={b} | expected={exp}")
    assert a == b == exp, "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

We time each approach on inputs of growing size `n` and read the **doubling ratio**.

| Theoretical | Ratio `n`→`2n` |
|---|---|
| `O(n)` / `O(V+E)` | ≈ **2×** |
| `O(n log n)`      | ≈ **2×** (slightly more) |
| `O(n²)`           | ≈ **4×** |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")): break
    _root = os.path.dirname(_root)
if _root not in sys.path: sys.path.insert(0, _root)
from bench_utils import benchmark

def make_worst_case(n):
    # Two far-apart leaves so both approaches traverse most of a balanced tree.
    return (build_balanced(n), 1, n)
solutions = {
    "paths     O(n)": lca_paths,
    "recursive O(n)": lca_recursive,
}
sizes = [1000, 2000, 4000, 8000]

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Return-something-up DFS:** a subtree reports "I contain a target"; a node seeing both children report is the answer — a reusable tree pattern (also used for max-path-sum, diameter).
- **General tree ≠ BST:** no ordering means you must search both sides (`O(n)`), not navigate (`O(height)`).
- **Signal:** "deepest common parent / where two nodes meet" in a non-ordered tree.
- **DevRev / related:** common parent of two tasks/comments; LCA in a workflow DAG; LeetCode 236 / 235 (BST) / 1650.
- **Common pitfalls:** (1) using the BST comparison trick on a general tree (wrong); (2) assuming a node can't be its own ancestor; (3) not handling a target equal to the current node.